> Graphs built out of signals and/or windows thereof to try and identify events of interest

In [ ]:
import pathlib

import pandas as pd

import plotly
from bokeh.io import output_notebook
from bokeh.resources import INLINE
from bokeh.palettes import Category20

from IPython.display import display, Markdown

import trapallada.plot
import trapallada.analysis

So that `import`ed Python modules are automatically reloaded when modified

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
output_notebook(resources=INLINE)

So that *plotly* pictures can be exported to HTML

In [ ]:
plotly.offline.init_notebook_mode()

The base directory

In [ ]:
base_dir = trapallada.analysis.get_base_directory()
base_dir

# Input data

Particular *sub*directory with the trained model. It should be a `str` so that parameters can be injected with papermill.

In [ ]:
# model_subdir = "results/arch/nouveau/time"
model_subdir = 'results/cvae/min_count_rate_gti_with_background/fft_vanilla_z_5'

Just in case

In [ ]:
model_dir = base_dir / model_subdir
display(model_dir)
assert model_dir.exists()

From the results directory,
- the `Dataset`,
- the headers of the corresponding files,
- the ML model,
- a `DataFrame` with results from training, 
- a `dict` with the parameters

Everything is loaded

In [ ]:
ds, positions_df, model, results_df, parameters, analysis_df = trapallada.analysis.load_results_dir(model_dir, analysis_also=True)

In [ ]:
data_dir = (pathlib.Path.home() / parameters['dataset']['files']['signals_file']).parent
data_dir

## Graph's post-processing

The mount point for the [Google Drive directory](https://drive.google.com/drive/u/0/folders/1nyxuRvEFIsEnAXVP0Z0wxJtJRLOPWB7R)

In [ ]:
googe_drive_dir = pathlib.Path.home() / 'drive' / 'git_supergraph' / 'projects'
display(googe_drive_dir)
assert googe_drive_dir.exists()

Particular subdirectory

In [ ]:
graph_subdir = 'starwVanillaZ05_00/graphs/sigsum/'

From the above *base directory* and *leave*, the final directory is assembled

In [ ]:
graph_dir = googe_drive_dir / graph_subdir
display(graph_dir)
assert graph_dir.exists()

All the files in the above directory

In [ ]:
[f.relative_to(graph_dir).as_posix() for f in graph_dir.glob('*')]

The files with information about the nodes and edges are *inferred*

In [ ]:
nodes_file = list(graph_dir.glob('*_nodes.csv'))[0]
edges_file = list(graph_dir.glob('*_edges.csv'))[0]
nodes_file.name, edges_file.name

Nodes are read

In [ ]:
nodes_df = pd.read_csv(nodes_file)
display(nodes_df.head()), nodes_df.shape

How many nodes in each community?

In [ ]:
nodes_df['leiden'].value_counts()

`main_type` for each signal is extracted from the *analysis* `DataFrame`. **UPDATE**: apparently, this is not needed anymore.

In [ ]:
# nodes_df = pd.merge(nodes_df, analysis_df[['signal', 'main_type']].drop_duplicates(), left_on='Id', right_on='signal', how='left')
# assert (nodes_df['Id'] == nodes_df['signal']).all()
# nodes_df.drop(columns='signal', inplace=True)
# display(nodes_df.head()), nodes_df.shape

In [ ]:
nodes_df['main_type'] == 'UNKNOWN'

It is used to determine the *marker* for each signal

In [ ]:
# unknown_series = nodes_df['main_type'].isna()
unknown_series = (nodes_df['main_type'] == 'UNKNOWN')
nodes_df.loc[unknown_series, 'marker'] = 'square'
nodes_df.loc[~unknown_series, 'marker'] = 'diamond'

More attributes and some renaming to allow plotting with the usual function

In [ ]:
palette = Category20[20]

nodes_df = nodes_df.rename(columns={'leiden': 'community'})
nodes_df['fill_color'] = [palette[c % 20]for c in nodes_df['community']]
nodes_df['size'] = 8
nodes_df['alpha'] = 1.
nodes_df.head()

`Id` (the signal identifier) is set as index (with **no name**, a requirement of the plotting function)

In [ ]:
nodes_df = nodes_df.set_index('Id')
nodes_df.index.name = None

Edges are read

In [ ]:
edges_df = pd.read_csv(edges_file)
edges_df.rename(columns={'Source': 'start', 'Target': 'end'}, inplace=True)
display(edges_df.head()), edges_df.shape

# Individual signals

Javi's paper, cataclysmic, ...

In [ ]:
ground_truth = trapallada.analysis.GroundTruth()
ground_truth

Signals of interest lost along the way...

In [ ]:
set(ground_truth.signals) - set(nodes_df.index)

In [ ]:
nodes_df.loc[list(set(nodes_df.index) & set(ground_truth.signals))]

# Graph

Plot of the graph along with *known* signals

In [ ]:
graph_plot = trapallada.analysis.SignalsGraph(nodes_df, edges_df, data_dir)
fig = graph_plot.make_fig()
graph_plot.highlight_nodes(ground_truth.signals_to_labels)

In [ ]:
# for signal, label in signals_of_interest.items():
#     display(Markdown(f'### {label}'))
#     graph_plot.display_selected(model, ds, signal)

## Regions

In [ ]:
regions = [
    {
        'x_range': (13000, 14000),
        'y_range': (9500, 10500),
    },
    {
        'x_range': (-1500, -500),
        'y_range': (-9600, -8400),
    },
    {
        'x_range': (-11500, -10500),
        'y_range': (-1500, 0),
    },
    {
        'x_range': (-5500, -3000),
        'y_range': (2500, 3500),
    },
]

graph_plot.inspect_regions(n_samples=10, regions=regions, model=model, ds=ds)